In [3]:
import numpy as np
import torch
import random
import copy
import hydra
from hydra import initialize, compose
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [4]:
from os.path import join, dirname
import os
import sys

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import cf

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO
from memoryPerceiver.memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


In [5]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(True)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [6]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
num_workers = 2
batch_size = 1

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [7]:
def create_model(device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params
    for part, params in cf.body_outputs.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_outputs)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = batch_size
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_outputs.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    queries=torch.cat(queries)


    return model, queries

In [8]:
model, queries = create_model()

inputs={}
inputs['video']=torch.randn(batch_size, 3, nb_frames, 256, 456)
inputs['audio']=torch.randn(batch_size, nb_frames*1920)
inputs['observations']=torch.randn(batch_size, nb_frames, 4)

batch = input_to_tensor(inputs, model.modality, cf.body_parts)

In [9]:
# with torch.no_grad():
#     x, aux_loss = model(batch.to(device), queries = queries)
#     output = {}
#     output['audio'] = rearrange(model.projections['audio'](x[:, :audio_future, :]) ,  "b t t1 -> b (t t1)")
#     start_modality = audio_future
#     for part, params in cf.body_outputs.items():
#         output[part] = model.projections[part](x[:, start_modality:start_modality+frames_future, :])
#         start_modality += frames_future

In [10]:
aa

NameError: name 'aa' is not defined

In [14]:
a=torch.randn(3,4)
a

tensor([[ 1.4026, -0.1058,  0.0098,  0.1424],
        [-1.0421, -1.1306,  0.4052,  1.1239],
        [ 1.0707, -0.8096, -0.3937, -0.3155]])

In [17]:
a[2:3]

tensor([[ 1.0707, -0.8096, -0.3937, -0.3155]])

In [18]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):

    cfg_dict = omegaconf_to_dict(cfg)
    print_dict(cfg_dict)

    headless = cfg.headless
    render = not headless

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    history = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[]}
    buffer = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[]}
    lifelong_reward = torch.zeros(numEnvs, 1)
    model.eval()
    while env._simulation_app.is_running():
        if env._world.is_playing():
            if env._world.current_time_step_index == 0:
                env._world.reset(soft=True)

            # Observations: 'cartpole_view' 'obs_buf' (numEnvs, 4)

            obs = env._task.get_observations()['cartpole_view']['obs_buf']
            buffer['obs'].append(obs)
            history['obs'].append(obs)
            if(len(buffer['obs'])>nb_frames):
                buffer['obs'].pop(0)

            # Reward (numEnvs, 1)

            env._task.calculate_metrics()
            reward=env._task.rew_buf
            buffer['reward'].append(reward)
            history['reward'].append(reward)
            if(len(buffer['reward'])>nb_frames):
                buffer['reward'].pop(0)

            # Process input

            inputs={}
            inputs['video']=torch.randn(numEnvs, 3, nb_frames, 256, 456)
            inputs['audio']=torch.randn(numEnvs, nb_frames*1920)
            inputs['observations']=rearrange(buffer['obs'], 't b d -> b t d')
            batch = input_to_tensor(inputs, model.modality, cf.body_parts)

            # Agent action

            output = {}
            if(len(buffer['action']) == 0 and len(buffer['obs']) >= nb_frames):
                new_actions = []
                for agent_i in range(numEnvs):
                    with torch.no_grad():
                        x, aux_loss = model(batch[agent_i:agent_i+1].to(device), queries = queries)
                        output['audio'] = rearrange(model.projections['audio'](
                            x[:, :audio_future, :]), "b t t1 -> b (t t1)"
                        )
                        start_modality = audio_future
                        for part, params in cf.body_outputs.items():
                            output[part] = model.projections[part](
                                x[:, start_modality:start_modality+frames_future, :]
                            )
                            start_modality += frames_future
                    new_actions.append(rearrange(output['actions'], '() t d -> t d'))
                new_actions = list(torch.unbind(
                                    rearrange(new_actions, 'b t d -> t b d')
                                ))
                buffer['action'] += new_actions
                history['action'] += new_actions

            # Save video, audio, body, reward
            
            if env.sim_frame_count % save_step == 0 and len(history['obs']) >= 0:
                print('OBS', buffer['obs'])
                print("$$$$$$$$$$$$ 0 &&& 0 $$$$$$$$$$$")
                print( 'REWARD', buffer['reward'])
                print("$$$$$$$$$$$$ 0 &&& 0 $$$$$$$$$$$")
                print( 'ACTION', buffer['action'])
                for key in history.keys():
                    history[key]=[]

            # Current action (numEnvs, 1)

            if(len(buffer['action'])>0):
                action = buffer['action'].pop(0)
            else:
                action = torch.randn(numEnvs, 1)

            # Step environment

            # action = torch.tensor(np.array([env.action_space.sample() for _ in range(env.num_envs)]), device=task.rl_device)
            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()
        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    save_step = 160
    numEnvs = 2
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


task: 
    name: Cartpole
    physics_engine: physx
    env: 
        numEnvs: 2
        envSpacing: 4.0
        resetDist: 3.0
        maxEffort: 400.0
        clipObservations: 5.0
        clipActions: 1.0
        controlFrequencyInv: 2
    sim: 
        dt: 0.0083
        use_gpu_pipeline: True
        gravity: [0.0, 0.0, -9.81]
        add_ground_plane: True
        add_distant_light: True
        use_flatcache: True
        enable_scene_query_support: False
        disable_contact_processing: False
        enable_cameras: False
        default_physics_material: 
            static_friction: 1.0
            dynamic_friction: 1.0
            restitution: 0.0
        physx: 
            worker_thread_count: 4
            solver_type: 1
            use_gpu: True
            solver_position_iteration_count: 4
            solver_velocity_iteration_count: 0
            contact_offset: 0.02
            rest_offset: 0.001
            bounce_threshold_velocity: 0.2
            friction_offs

/isaac-sim/exts/omni.isaac.ml_archive/pip_prebundle/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


] startup
[4.162s] [ext: omni.replicator.core-1.6.4] startup
2023-02-23 09:14:51 [4,492ms] [Warning] [omni.replicator.core.scripts.annotators] Annotator PostProcessDispatch is already registered, overwriting annotator template
[4.726s] [ext: omni.isaac.core-1.42.0] startup
[4.876s] [ext: omni.graph.bundle.action-1.3.0] startup
[4.877s] [ext: omni.replicator.isaac-1.7.3] startup
[4.906s] [ext: omni.kit.window.title-1.1.2] startup
2023-02-23 09:14:52 [4,897ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-02-23 09:14:52 [4,897ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2023-02-23 09:14:52 [4,898ms] [Warning] [carb] [Plugin: libomni.structuredlog.plugin.so] Module /isaac-sim/kit/libomni.structuredlog.plugin.so remained loaded after unload request
2023-02-23 09:14:52 [4,900ms] [Warning] [carb.windowing-glfw.plugin] GLF

2023-02-23 09:26:31 [703,967ms] [Error] [carb.scripting-python.plugin] AttributeError: 'NoneType' object has no attribute 'deregister_hotkey'

At:
  /isaac-sim/kit/extscore/omni.kit.menu.utils/omni/kit/menu/utils/scripts/debug_window.py(48): on_shutdown
  /isaac-sim/kit/kernel/py/omni/ext/_impl/_internal.py(207): shutdown
  /isaac-sim/kit/kernel/py/omni/ext/_impl/_internal.py(288): shutdown_extension
  PythonExtension.cpp::shutdown()(2): <module>
  /isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py(472): close
  /tmp/ipykernel_208743/1519359505.py(104): parse_hydra_configs
  /tmp/ipykernel_208743/1519359505.py(112): <module>
  /isaac-sim/kit/python/lib/python3.7/site-packages/IPython/core/interactiveshell.py(3553): run_code
  /isaac-sim/kit/python/lib/python3.7/site-packages/IPython/core/interactiveshell.py(3473): run_ast_nodes
  /isaac-sim/kit/python/lib/python3.7/site-packages/IPython/core/interactiveshell.py(3258): run_cell_async
  /isaac-sim/kit/python/lib/python3.7/si

[703.982s] [ext: omni.physx-104.1.6-5.1] shutdown
2023-02-23 09:26:31 [703,970ms] [Warning] [carb] Plugin interface for a client: carb.scripting-python.plugin was already released.
[703.982s] [ext: omni.kit.window.title-1.1.2] shutdown
2023-02-23 09:26:31 [704,002ms] [Warning] [omni.ext._impl._internal] omni.kit.window.title-1.1.2 -> <class 'omni.kit.window.title.title.WindowTitle'>: extension object is still alive, something holds a reference on it. References: ["[0]:type: <class 'frame'>, id: 1201944752"]
[704.014s] [ext: omni.debugdraw-0.1.1] shutdown
[704.014s] [ext: omni.graph.core-2.64.7] shutdown
[704.014s] [ext: omni.kit.usd.layers-2.0.10] shutdown
[704.016s] [ext: omni.kit.widget.viewport-104.1.15] shutdown
[704.016s] [ext: omni.kit.hydra_texture-1.0.11] shutdown
[704.017s] [ext: omni.usd-1.6.23] shutdown
2023-02-23 09:26:31 [704,277ms] [Warning] [carb.assets.plugin] Evicting asset type c1237a43d569db21 v0.1 orphaned snapshot handle 4294967296 (/tmp/carb.lwYk0m/navtree.40x40.p

2023-02-23 09:26:32 [704,919ms] [Error] [carb.graphics-vulkan.plugin] pipelineLayouts: 2 or descriptorSetLayouts: 2 is leaking memory. Missing call to destroyResourceBindingSignature()


: 

: 

In [ ]:
%tb

No traceback available to show.
